# Ray Job Submission Tutorial

A short guide to running a `checkmaite` capability asynchronously with the default registry-backed `ray` job backend.

## When to use this backend

Use `ray` when jobs should be tracked through shared Ray actors so clients can list, fetch, and reattach to jobs while the Ray cluster remains alive.

Important assumptions:

- `idempotency_scope` is required and should identify your workspace or project;
- the registry/controller actors live in the Ray cluster, not in the notebook process;
- completed results are written to the configured analytics store before `job.result()` succeeds.

## Setup

In [ ]:
import tempfile
import uuid
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalCleaning
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset
from checkmaite.jobs import (
    configure_job_backend,
    get_job,
    list_jobs,
    shutdown_job_backend,
    submit_capability,
)

# Find the repository root whether the notebook is run from docs/tool-usage or elsewhere.
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())

# Use the tiny COCO fixture included with the repository.
dataset_root = REPO_ROOT / "tests/data_for_tests/coco_dataset"
dataset_ann = dataset_root / "ann_file.json"

dataset = CocoDetectionDataset(
    root=str(dataset_root),
    ann_file=str(dataset_ann),
    dataset_id="coco-job-tutorial",
)

store_dir = Path(tempfile.mkdtemp(prefix="checkmaite_jobs_")) / "analytics_store"
analytics_store_config = {"backend": "parquet", "uri": str(store_dir)}

print(f"Dataset: {dataset.metadata['id']} ({len(dataset)} images)")
print(f"Analytics store: {store_dir}")

## Configure the default `ray` backend

This local tutorial uses a unique `idempotency_scope` for isolation. In a shared environment, choose a stable scope for the project or workspace.

In [ ]:
idempotency_scope = f"tool-usage-ray-{uuid.uuid4().hex}"

configure_job_backend(
    "ray",
    address="local",
    force_reinit=True,
    idempotency_scope=idempotency_scope,
    analytics_store=analytics_store_config,
    # Keep the local tutorial cluster tidy after terminal state is committed.
    controller_retention_s=0.0,
    max_retained_terminal_controllers=0,
)

print(f"Configured ray job backend with scope: {idempotency_scope}")

## Submit a capability job

In [ ]:
capability = DataevalCleaning()

job = submit_capability(
    capability,
    datasets=[dataset],
    use_cache=False,
)

print(f"Job ID: {job.job_id}")
print(f"Initial status: {job.status}")

## Wait for completion and get the result reference

`job.result()` returns a `CapabilityRunRef`, not the full capability output object.

In [ ]:
final_status = job.wait(timeout=120)
print(f"Final status: {final_status}")

ref = job.result(timeout=10)
print(f"Run UID: {ref.run_uid}")
print(f"Store URI: {ref.store_uri}")

## List and fetch tracked jobs

The default `ray` backend stores job metadata in the shared registry actor, so `list_jobs()` and `get_job(...)` read registry-backed state.

In [ ]:
for tracked_job in list_jobs(limit=10):
    print(tracked_job.job_id, tracked_job.status)

fetched = get_job(job.job_id)
print("Fetched by ID:", fetched.job_id, fetched.status)

## Duplicate submission in the same scope

Submitting the same logical run again in the same `idempotency_scope` returns the existing tracked job instead of launching duplicate work.

In [ ]:
same_job = submit_capability(
    capability,
    datasets=[dataset],
    use_cache=False,
)

print(f"Original job:  {job.job_id}")
print(f"Second submit: {same_job.job_id}")
print("Same tracked job:", same_job.job_id == job.job_id)

## Query the analytics store

In [ ]:
store = AnalyticsStore(ParquetBackend(str(store_dir)))

print(f"Tables: {store.list_tables()}")

cleaning_results = store.query_sql("""
    SELECT
        dataset_id,
        exact_duplicate_count,
        image_outlier_count,
        image_outlier_ratio,
        mean_brightness
    FROM dataeval_cleaning
""")
print(cleaning_results)

## Shut down

In [ ]:
shutdown_job_backend(wait=True)
print("Job backend shut down.")